# 00. Configuration

In [12]:
!pip install wandb -q

import warnings
warnings.filterwarnings('ignore')

In [13]:
# ============================================================
# CUHK-X MULTIMODAL HUMAN ACTIVITY RECOGNITION
# Small Model Track
# ============================================================

from pathlib import Path

# -----------------------------
# Dataset
# -----------------------------

BASE_DATA_ROOT = Path(
    "/kaggle/input/datasets/samasiayushman/small-model-track/"
    "Training/Training/data"
)

# Modalities
SKELETON_ROOT = BASE_DATA_ROOT / "Skeleton"
IMU_ROOT = BASE_DATA_ROOT / "IMU"
IR_ROOT = BASE_DATA_ROOT / "IR"
DEPTH_ROOT = BASE_DATA_ROOT / "Depth_Color"
RADAR_ROOT = BASE_DATA_ROOT / "Radar"
THERMAL_ROOT = BASE_DATA_ROOT / "Thermal"


# -----------------------------
# Competition setup
# -----------------------------

NUM_CLASSES = 40

TRAIN_USERS = {
    "user1", "user2", "user3", "user4",
    "user5", "user6", "user7", "user8",
    "user9",

    "user16", "user17", "user18", "user19",
    "user20", "user21", "user22", "user23",
    "user24",
}

# Our fixed local validation subjects
VAL_USERS = {
    "user8",
    "user9",
    "user23",
    "user24",
}

TRAIN_USERS_LOCAL = TRAIN_USERS - VAL_USERS


# Official competition test subjects
TEST_USERS = {
    "user10",
    "user11",
    "user25",
    "user26",
}


# -----------------------------
# Sequence configuration
# -----------------------------

SEQUENCE_LENGTH = 64

# Visual modalities
VISUAL_HEIGHT = 112
VISUAL_WIDTH = 112


# -----------------------------
# Training
# -----------------------------

BATCH_SIZE = 32
NUM_WORKERS = 2

SEED = 42

print("Base data root:", BASE_DATA_ROOT)
print("Number of classes:", NUM_CLASSES)

print("\nTraining users:", len(TRAIN_USERS))
print("Local validation users:", sorted(VAL_USERS))
print("Official test users:", sorted(TEST_USERS))

print("\nDataset roots:")
for name, path in {
    "Skeleton": SKELETON_ROOT,
    "IMU": IMU_ROOT,
    "IR": IR_ROOT,
    "Depth_Color": DEPTH_ROOT,
    "Radar": RADAR_ROOT,
    "Thermal": THERMAL_ROOT,
}.items():
    print(f"{name:12s} -> {path}")

Base data root: /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data
Number of classes: 40

Training users: 18
Local validation users: ['user23', 'user24', 'user8', 'user9']
Official test users: ['user10', 'user11', 'user25', 'user26']

Dataset roots:
Skeleton     -> /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Skeleton
IMU          -> /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/IMU
IR           -> /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/IR
Depth_Color  -> /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Depth_Color
Radar        -> /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Radar
Thermal      -> /kaggle/input/datasets/samasiayushman/small-model-track/Training/Training/data/Thermal


#  01. Imports & Reproducibility

In [14]:
import os
import json
import random
import math
import time
import gc

import numpy as np
import pandas as pd

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt


# -----------------------------
# Reproducibility
# -----------------------------

def seed_everything(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    # Reproducible behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(SEED)


# -----------------------------
# Device
# -----------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("PyTorch:", torch.__version__)
print("Device:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.10.0+cpu
Device: cpu


## i. Wandb Setup

In [15]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

# Fetch the secret token safely
user_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = user_secrets.get_secret("WANDB_API_KEY")

# Log into WandB
wandb.login()

True

# 02. Dataset Discovery

In [16]:
# ============================================================
# DATASET DISCOVERY
# ============================================================

def get_action_dirs(root):

    return sorted([
        p for p in root.iterdir()
        if p.is_dir()
    ])


action_dirs = get_action_dirs(SKELETON_ROOT)

print("Number of action directories:", len(action_dirs))

print("\nFirst 10 actions:")
for p in action_dirs[:10]:
    print(p.name)

Number of action directories: 40

First 10 actions:
0_Wash_face
10_Stir_drinks
11_Peel_fruits
12_Sweep_the_floor
13_Mop_the_floor
14_Wipe_bowls
15_Wipe_windows_and_tables
16_Fold_clothes
17_Tap_the_keyboard
18_Write


In [17]:
# ============================================================
# BUILD COMMON TRIAL INDEX
# ============================================================

records = []

for action_dir in action_dirs:

    action = action_dir.name

    for user_dir in sorted(action_dir.iterdir()):

        if not user_dir.is_dir():
            continue

        user = user_dir.name

        for trial_dir in sorted(user_dir.iterdir()):

            if not trial_dir.is_dir():
                continue

            trial = trial_dir.name

            records.append({
                "action": action,
                "user": user,
                "trial": trial,
            })


trial_df = pd.DataFrame(records)

print("Total Skeleton trials:", len(trial_df))
print("Actions:", trial_df["action"].nunique())
print("Users:", trial_df["user"].nunique())

print("\nUsers:")
print(sorted(trial_df["user"].unique()))

print("\nExample:")
display(trial_df.head())

Total Skeleton trials: 2931
Actions: 40
Users: 18

Users:
['user1', 'user16', 'user17', 'user18', 'user19', 'user2', 'user20', 'user21', 'user22', 'user23', 'user24', 'user3', 'user4', 'user5', 'user6', 'user7', 'user8', 'user9']

Example:


,action,user,trial
0,0_Wash_face,user16,1-1-1
1,0_Wash_face,user16,1-1-2
2,0_Wash_face,user16,1-1-3
3,0_Wash_face,user18,7-1-1
4,0_Wash_face,user18,7-1-2


# 03. Common Trial Index

In [18]:
def check_modality_trial(root, action, user, trial):
    """
    Check whether a modality contains this exact
    action / user / trial directory.
    """
    trial_dir = root / action / user / trial
    return trial_dir.exists()


modality_roots = {
    "skeleton": SKELETON_ROOT,
    "imu": IMU_ROOT,
    "ir": IR_ROOT,
    "depth": DEPTH_ROOT,
    "radar": RADAR_ROOT,
    "thermal": THERMAL_ROOT,
}


multimodal_df = trial_df.copy()

for modality, root in modality_roots.items():

    multimodal_df[modality] = [
        check_modality_trial(
            root,
            row["action"],
            row["user"],
            row["trial"]
        )
        for _, row in multimodal_df.iterrows()
    ]


print("Total trials:", len(multimodal_df))

print("\nModality availability:")
for modality in modality_roots:
    count = multimodal_df[modality].sum()
    percentage = 100 * count / len(multimodal_df)

    print(
        f"{modality:10s}: "
        f"{count:4d} / {len(multimodal_df)} "
        f"({percentage:.2f}%)"
    )

Total trials: 2931

Modality availability:
skeleton  : 2931 / 2931 (100.00%)
imu       : 2903 / 2931 (99.04%)
ir        : 2931 / 2931 (100.00%)
depth     : 2931 / 2931 (100.00%)
radar     : 2914 / 2931 (99.42%)
thermal   : 2786 / 2931 (95.05%)


## A. SHOW TRIAL AVAILABILITY


In [19]:
display(multimodal_df.head(10))

,action,user,trial,skeleton,imu,ir,depth,radar,thermal
0,0_Wash_face,user16,1-1-1,True,True,True,True,True,True
1,0_Wash_face,user16,1-1-2,True,True,True,True,True,True
2,0_Wash_face,user16,1-1-3,True,True,True,True,True,True
3,0_Wash_face,user18,7-1-1,True,True,True,True,True,True
4,0_Wash_face,user18,7-1-2,True,True,True,True,True,True
5,0_Wash_face,user18,7-1-3,True,True,True,True,True,True
6,0_Wash_face,user20,4-2-1,True,True,True,True,True,True
7,0_Wash_face,user20,4-2-2,True,True,True,True,True,True
8,0_Wash_face,user20,4-2-3,True,True,True,True,True,True
9,0_Wash_face,user21,1-1-1,True,True,True,True,True,True


## B. MISSING MODALITY COUNTS


In [20]:
for modality in modality_roots:

    missing = (~multimodal_df[modality]).sum()

    print(
        f"{modality:10s} missing trials: {missing}"
    )

skeleton   missing trials: 0
imu        missing trials: 28
ir         missing trials: 0
depth      missing trials: 0
radar      missing trials: 17
thermal    missing trials: 145


## C. FIXED SUBJECT-BASED TRAIN / VALIDATION SPLIT


In [21]:
local_train_df = multimodal_df[
    multimodal_df["user"].isin(TRAIN_USERS_LOCAL)
].reset_index(drop=True)

local_val_df = multimodal_df[
    multimodal_df["user"].isin(VAL_USERS)
].reset_index(drop=True)


print("LOCAL TRAIN")
print("Trials:", len(local_train_df))
print("Users:", sorted(local_train_df["user"].unique()))

print("\nLOCAL VALIDATION")
print("Trials:", len(local_val_df))
print("Users:", sorted(local_val_df["user"].unique()))

LOCAL TRAIN
Trials: 2295
Users: ['user1', 'user16', 'user17', 'user18', 'user19', 'user2', 'user20', 'user21', 'user22', 'user3', 'user4', 'user5', 'user6', 'user7']

LOCAL VALIDATION
Trials: 636
Users: ['user23', 'user24', 'user8', 'user9']


## D. LEAKAGE CHECK


In [22]:
train_subjects = set(local_train_df["user"])
val_subjects = set(local_val_df["user"])

overlap = train_subjects & val_subjects

print("Train subjects:", sorted(train_subjects))
print("Val subjects:", sorted(val_subjects))
print("Subject overlap:", overlap)

assert len(overlap) == 0, "SUBJECT LEAKAGE DETECTED!"

Train subjects: ['user1', 'user16', 'user17', 'user18', 'user19', 'user2', 'user20', 'user21', 'user22', 'user3', 'user4', 'user5', 'user6', 'user7']
Val subjects: ['user23', 'user24', 'user8', 'user9']
Subject overlap: set()


## E. CLASS DISTRIBUTION


In [23]:
train_counts = local_train_df["action"].value_counts().sort_index()
val_counts = local_val_df["action"].value_counts().sort_index()

class_distribution = pd.DataFrame({
    "train": train_counts,
    "validation": val_counts
})

display(class_distribution)

print(
    "\nTrain min/max:",
    train_counts.min(),
    train_counts.max()
)

print(
    "Val min/max:",
    val_counts.min(),
    val_counts.max()
)

,train,validation
action,,
0_Wash_face,29,15.0
10_Stir_drinks,98,19.0
11_Peel_fruits,91,15.0
12_Sweep_the_floor,51,12.0
13_Mop_the_floor,48,8.0
14_Wipe_bowls,32,3.0
15_Wipe_windows_and_tables,40,12.0
16_Fold_clothes,15,9.0
17_Tap_the_keyboard,64,22.0



Train min/max: 6 256
Val min/max: 3 79


# 04. Train / Validation Split

## F0: SKELETON + IMU AVAILABILITY


In [24]:

f0_df = multimodal_df[
    multimodal_df["skeleton"] &
    multimodal_df["imu"]
].copy().reset_index(drop=True)

f0_train_df = f0_df[
    f0_df["user"].isin(TRAIN_USERS_LOCAL)
].reset_index(drop=True)

f0_val_df = f0_df[
    f0_df["user"].isin(VAL_USERS)
].reset_index(drop=True)

print("F0 — Skeleton + IMU")
print("=" * 50)

print(f"Total common trials: {len(f0_df)}")
print(f"Train trials:        {len(f0_train_df)}")
print(f"Validation trials:   {len(f0_val_df)}")

print("\nTrain users:")
print(sorted(f0_train_df["user"].unique()))

print("\nValidation users:")
print(sorted(f0_val_df["user"].unique()))

F0 — Skeleton + IMU
Total common trials: 2903
Train trials:        2267
Validation trials:   636

Train users:
['user1', 'user16', 'user17', 'user18', 'user19', 'user2', 'user20', 'user21', 'user22', 'user3', 'user4', 'user5', 'user6', 'user7']

Validation users:
['user23', 'user24', 'user8', 'user9']


In [25]:
# ============================================================
# F0 CLASS DISTRIBUTION
# ============================================================

f0_train_counts = f0_train_df["action"].value_counts().sort_index()
f0_val_counts = f0_val_df["action"].value_counts().sort_index()

f0_distribution = pd.DataFrame({
    "train": f0_train_counts,
    "validation": f0_val_counts
})

display(f0_distribution)

print("\nTrain min/max:",
      f0_train_counts.min(),
      f0_train_counts.max())

print("Val min/max:",
      f0_val_counts.min(),
      f0_val_counts.max())


,train,validation
action,,
0_Wash_face,29,15.0
10_Stir_drinks,96,19.0
11_Peel_fruits,90,15.0
12_Sweep_the_floor,51,12.0
13_Mop_the_floor,45,8.0
14_Wipe_bowls,32,3.0
15_Wipe_windows_and_tables,37,12.0
16_Fold_clothes,15,9.0
17_Tap_the_keyboard,64,22.0



Train min/max: 6 251
Val min/max: 3 79


# 04 IR Model

# 04 Depth Color

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
records = []
ROOT = Path(
    "/kaggle/input/datasets/samasiayushman/"
    "small-model-track/Training/Training/data/Depth_Color"
)
for action_dir in sorted(ROOT.iterdir()):

    if not action_dir.is_dir():
        continue

    # Example:
    # 0_Wash_face -> 0
    label = int(
        action_dir.name.split("_", 1)[0]
    )

    for user_dir in sorted(action_dir.iterdir()):

        if not user_dir.is_dir():
            continue

        user = user_dir.name

        for trial_dir in sorted(user_dir.iterdir()):

            if not trial_dir.is_dir():
                continue

            frames = sorted(
                trial_dir.glob("*.png")
            )

            if len(frames) == 0:
                continue

            records.append({
                "action": action_dir.name,
                "label": label,
                "user": user,
                "trial": trial_dir.name,
                "path": str(trial_dir),
                "num_frames": len(frames)
            })


df = pd.DataFrame(records)

print("Total trials:", len(df))
print("Classes:", df["label"].nunique())
print("Users:", df["user"].nunique())

display(df.head())
users = np.array(
    sorted(df["user"].unique())
)

print("Total users:", len(users))
print(users)

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, val_idx = next(
    gss.split(
        df,
        groups=df["user"]
    )
)

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

print("Train trials:", len(train_df))
print("Val trials:", len(val_df))

print(
    "Train users:",
    train_df["user"].nunique()
)

print(
    "Val users:",
    val_df["user"].nunique()
)

# Absolutely verify no leakage
assert set(train_df["user"]).isdisjoint(
    set(val_df["user"])
)



In [ ]:
# ============================================================
# EXACT JET LOOKUP
# ============================================================
import cv2
gray = np.arange(
    256,
    dtype=np.uint8
).reshape(-1, 1)

jet_bgr = cv2.applyColorMap(
    gray,
    cv2.COLORMAP_JET
)

jet_rgb = (
    jet_bgr[:, :, ::-1]
    .reshape(256, 3)
)

jet_keys = (
    (jet_rgb[:, 0].astype(np.int32) << 16)
    | (jet_rgb[:, 1].astype(np.int32) << 8)
    | jet_rgb[:, 2].astype(np.int32)
)

RGB_TO_DEPTH = np.full(
    256 ** 3,
    -1,
    dtype=np.int16
)

RGB_TO_DEPTH[jet_keys] = np.arange(
    256,
    dtype=np.int16
)

print(
    "Lookup memory:",
    RGB_TO_DEPTH.nbytes / 1024**2,
    "MB"
)

print("JET colors:", len(jet_keys))

In [ ]:
def decode_depth(path):

    rgb = np.asarray(
        Image.open(path).convert("RGB"),
        dtype=np.uint8
    )

    # RGB → packed integer
    key = (
        (rgb[:, :, 0].astype(np.int32) << 16)
        | (rgb[:, :, 1].astype(np.int32) << 8)
        | rgb[:, :, 2].astype(np.int32)
    )

    depth_uint8 = RGB_TO_DEPTH[key]

    # Valid JET pixels
    valid = depth_uint8 >= 0

    depth = np.maximum(
        depth_uint8,
        0
    ).astype(np.float32) / 255.0

    mask = valid.astype(np.float32)

    return depth, mask

In [ ]:
test_path = Path(
    df.iloc[0]["path"]
)

test_frame = sorted(
    test_path.glob("*.png")
)[0]

depth, mask = decode_depth(
    test_frame
)

print("Frame:", test_frame.name)
print("Depth:", depth.shape, depth.dtype)

print(
    "Depth min:",
    depth[mask > 0].min()
)

print(
    "Depth max:",
    depth[mask > 0].max()
)

print(
    "Depth mean:",
    depth[mask > 0].mean()
)

print(
    "Depth std:",
    depth[mask > 0].std()
)

print(
    "Valid ratio:",
    mask.mean()
)

def sample_frames(
    frame_paths,
    max_frames=48
):

    n = len(frame_paths)

    if n <= max_frames:
        return frame_paths

    indices = np.linspace(
        0,
        n - 1,
        max_frames
    ).round().astype(int)

    return [
        frame_paths[i]
        for i in indices
    ]

In [ ]:
class DepthColorDataset(Dataset):

    def __init__(
        self,
        dataframe,
        image_size=192,
        max_frames=48
    ):

        self.df = dataframe.reset_index(
            drop=True
        )

        self.image_size = image_size
        self.max_frames = max_frames

    def __len__(self):

        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        trial_dir = Path(
            row["path"]
        )

        frame_paths = sorted(
            trial_dir.glob("*.png")
        )

        frame_paths = sample_frames(
            frame_paths,
            self.max_frames
        )

        frames = []

        for frame_path in frame_paths:

            depth, valid = decode_depth(
                frame_path
            )

            # -----------------------------------------
            # numpy → torch
            # -----------------------------------------

            depth = torch.from_numpy(
                depth
            ).unsqueeze(0)

            valid = torch.from_numpy(
                valid
            ).unsqueeze(0)

            # -----------------------------------------
            # Resize depth
            # -----------------------------------------

            depth = F.interpolate(
                depth.unsqueeze(0),
                size=(
                    self.image_size,
                    self.image_size
                ),
                mode="bilinear",
                align_corners=False
            ).squeeze(0)

            # -----------------------------------------
            # Resize validity mask
            # -----------------------------------------

            valid = F.interpolate(
                valid.unsqueeze(0),
                size=(
                    self.image_size,
                    self.image_size
                ),
                mode="nearest"
            ).squeeze(0)

            # -----------------------------------------
            # D0 representation
            #
            # [2,H,W]
            #
            # depth
            # valid mask
            # -----------------------------------------

            frame = torch.cat(
                [
                    depth,
                    valid
                ],
                dim=0
            )

            frames.append(frame)

        x = torch.stack(frames)

        # [T, 2, H, W]

        y = torch.tensor(
            row["label"],
            dtype=torch.long
        )

        return x, y

In [ ]:
def depth_collate(batch):

    xs, ys = zip(*batch)

    max_t = max(
        x.shape[0]
        for x in xs
    )

    batch_size = len(xs)

    C = xs[0].shape[1]
    H = xs[0].shape[2]
    W = xs[0].shape[3]

    x_pad = torch.zeros(
        batch_size,
        max_t,
        C,
        H,
        W,
        dtype=torch.float32
    )

    time_mask = torch.zeros(
        batch_size,
        max_t,
        dtype=torch.bool
    )

    for i, x in enumerate(xs):

        t = x.shape[0]

        x_pad[i, :t] = x
        time_mask[i, :t] = True

    y = torch.stack(ys)

    return (
        x_pad,
        time_mask,
        y
    )

In [ ]:
IMAGE_SIZE = 128
MAX_FRAMES = 48

train_dataset = DepthColorDataset(
    train_df,
    image_size=IMAGE_SIZE,
    max_frames=MAX_FRAMES
)

val_dataset = DepthColorDataset(
    val_df,
    image_size=IMAGE_SIZE,
    max_frames=MAX_FRAMES
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=depth_collate
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=depth_collate
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

x, time_mask, y = next(
    iter(train_loader)
)

print("X:", x.shape)
print("Time mask:", time_mask.shape)
print("Lengths:", time_mask.sum(dim=1).tolist())
print("Labels:", y.tolist())

In [ ]:
class DepthSpatialEncoder(nn.Module):

    def __init__(self, out_dim=256):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                2, 32,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(32),
            nn.GELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                32, 64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(64),
            nn.GELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                64, 128,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(128),
            nn.GELU(),
            nn.MaxPool2d(2),

            nn.Conv2d(
                128, 256,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm2d(256),
            nn.GELU(),

            nn.AdaptiveAvgPool2d(1)
        )

        self.projection = nn.Linear(
            256,
            out_dim
        )

    def forward(self, x):

        x = self.features(x)

        x = x.flatten(1)

        x = self.projection(x)

        return x

In [ ]:
class TemporalAttention(nn.Module):

    def __init__(
        self,
        dim=256,
        heads=4
    ):
        super().__init__()

        self.attention = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=heads,
            batch_first=True
        )

        self.norm = nn.LayerNorm(dim)

        self.score = nn.Linear(
            dim,
            1
        )

    def forward(
        self,
        x,
        valid_mask
    ):

        # ------------------------------------------------
        # x: [B, T, D]
        # valid_mask: [B, T]
        # ------------------------------------------------

        attn_out, _ = self.attention(
            x,
            x,
            x,
            key_padding_mask=~valid_mask
        )

        x = self.norm(
            x + attn_out
        )

        scores = self.score(
            x
        ).squeeze(-1)

        # ------------------------------------------------
        # IMPORTANT:
        # Don't use -1e9 under FP16.
        # ------------------------------------------------

        scores = scores.masked_fill(
            ~valid_mask,
            torch.finfo(scores.dtype).min
        )

        weights = torch.softmax(
            scores,
            dim=1
        )

        pooled = torch.sum(
            x * weights.unsqueeze(-1),
            dim=1
        )

        return pooled

In [ ]:
class DepthD0(nn.Module):

    def __init__(
        self,
        num_classes=40
    ):

        super().__init__()

        # ---------------------------------------------
        # Spatial
        # ---------------------------------------------

        self.spatial = DepthSpatialEncoder(
            out_dim=256
        )

        # ---------------------------------------------
        # Temporal
        # ---------------------------------------------

        self.temporal = nn.LSTM(
            input_size=256,
            hidden_size=128,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.3
        )

        # BiLSTM:
        # 128 × 2 = 256
        # ---------------------------------------------

        self.attention = TemporalAttention(
            dim=256,
            heads=4
        )

        # ---------------------------------------------
        # Classifier
        # ---------------------------------------------

        self.classifier = nn.Sequential(

            nn.Linear(
                256,
                128
            ),

            nn.LayerNorm(128),

            nn.GELU(),

            nn.Dropout(0.3),

            nn.Linear(
                128,
                num_classes
            )
        )

    def forward(
        self,
        x,
        valid_mask
    ):

        B, T, C, H, W = x.shape

        # ------------------------------------------------
        # Flatten temporal dimension
        # ------------------------------------------------

        flat_x = x.reshape(
            B * T,
            C,
            H,
            W
        )

        flat_mask = valid_mask.reshape(-1)
        real_x = flat_x[flat_mask]
        CHUNK_SIZE = 4
        feature_chunks = []
        for i in range(0, real_x.size(0), CHUNK_SIZE):
            chunk = real_x[i:i + CHUNK_SIZE]
            chunk_features = self.spatial(chunk)
            feature_chunks.append(chunk_features)
        real_features = torch.cat(feature_chunks, dim=0)

        # ------------------------------------------------
        # Put features back
        # ------------------------------------------------

        features = torch.zeros(
            B * T,
            256,
            device=x.device,
            dtype=real_features.dtype
        )

        features[
            flat_mask
        ] = real_features

        features = features.reshape(
            B,
            T,
            256
        )

        # ------------------------------------------------
        # Packed LSTM
        # ------------------------------------------------

        lengths = (
            valid_mask
            .sum(dim=1)
            .cpu()
        )

        packed = nn.utils.rnn.pack_padded_sequence(
            features,
            lengths,
            batch_first=True,
            enforce_sorted=False
        )

        packed_out, _ = self.temporal(
            packed
        )

        temporal_out, _ = (
            nn.utils.rnn
            .pad_packed_sequence(
                packed_out,
                batch_first=True,
                total_length=T
            )
        )

        # ------------------------------------------------
        # Attention pooling
        # ------------------------------------------------

        embedding = self.attention(
            temporal_out,
            valid_mask
        )

        # ------------------------------------------------
        # Classifier
        # ------------------------------------------------

        logits = self.classifier(
            embedding
        )

        return logits, embedding

In [ ]:
model = DepthD0(
    num_classes=NUM_CLASSES
).to(DEVICE)

print(model)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

print(
    f"Total parameters: "
    f"{total_params:,}"
)

print(
    f"Trainable parameters: "
    f"{trainable_params:,}"
)
x, time_mask, y = next(
    iter(train_loader)
)

x = x.to(DEVICE)
time_mask = time_mask.to(DEVICE)
y = y.to(DEVICE)

with torch.no_grad():

    logits, embedding = model(
        x,
        time_mask
    )

print("Input:", x.shape)
print("Mask:", time_mask.shape)
print("Labels:", y.shape)

print("Logits:", logits.shape)
print("Embedding:", embedding.shape)

In [ ]:
criterion = nn.CrossEntropyLoss()
LR = 1e-3
WEIGHT_DECAY = 1e-4
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=3
)

print(optimizer)
print(scheduler)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

In [ ]:
def train_one_epoch(
    model,
    loader,
    criterion,
    optimizer,
    scaler,
    device
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for x, time_mask, y in loader:

        x = x.to(
            device,
            non_blocking=True
        )

        time_mask = time_mask.to(
            device,
            non_blocking=True
        )

        y = y.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):

            logits, _ = model(
                x,
                time_mask
            )

            loss = criterion(
                logits,
                y
            )

        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        running_loss += (
            loss.item()
            * y.size(0)
        )

        predictions = logits.argmax(
            dim=1
        )

        correct += (
            predictions == y
        ).sum().item()

        total += y.size(0)

    epoch_loss = (
        running_loss / total
    )

    epoch_acc = (
        correct / total
    )

    return epoch_loss, epoch_acc

@torch.no_grad()
def validate(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    all_predictions = []
    all_targets = []

    for x, time_mask, y in loader:

        x = x.to(
            device,
            non_blocking=True
        )

        time_mask = time_mask.to(
            device,
            non_blocking=True
        )

        y = y.to(
            device,
            non_blocking=True
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):

            logits, _ = model(
                x,
                time_mask
            )

            loss = criterion(
                logits,
                y
            )

        running_loss += (
            loss.item()
            * y.size(0)
        )

        predictions = logits.argmax(
            dim=1
        )

        correct += (
            predictions == y
        ).sum().item()

        total += y.size(0)

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_targets.extend(
            y.cpu().numpy()
        )

    epoch_loss = (
        running_loss / total
    )

    epoch_acc = (
        correct / total
    )

    return (
        epoch_loss,
        epoch_acc,
        np.array(all_targets),
        np.array(all_predictions)
    )

In [ ]:
best_val_acc = -1.0
best_epoch = -1

history = []

CHECKPOINT_PATH = "/kaggle/working/depth_D0_best.pth"


for epoch in range(1, 10):

    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        scaler,
        DEVICE
    )

    val_loss, val_acc, targets, predictions = validate(
        model,
        val_loader,
        criterion,
        DEVICE
    )

    scheduler.step(
        val_acc
    )

    current_lr = optimizer.param_groups[0]["lr"]

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
        "lr": current_lr
    })

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"LR: {current_lr:.2e}"
    )

    # ------------------------------------------------
    # Save BEST model
    # ------------------------------------------------

    if val_acc > best_val_acc:

        best_val_acc = val_acc
        best_epoch = epoch

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "val_acc": val_acc,
                "val_loss": val_loss
            },
            CHECKPOINT_PATH
        )

        print(
            f"🔥 New best: "
            f"{val_acc:.4f}"
        )


print()
print("================================")
print("D0 TRAINING COMPLETE")
print("================================")
print(
    f"Best Val Acc: {best_val_acc:.4f}"
)
print(
    f"Best Epoch: {best_epoch}"
)
print(
    f"Checkpoint: {CHECKPOINT_PATH}"
)

# 05. Skeleton Dataset

In [ ]:
SKELETON_SEQUENCE_LENGTH = 64
SKELETON_NUM_JOINTS = 17
SKELETON_FEATURES_PER_JOINT = 12

# COCO-17 parent structure
SKELETON_PARENTS = [
    -1, 0, 0, 1, 2,
    11, 12,
    5, 6,
    7, 8,
    -1, -1,
    11, 12,
    13, 14
]

In [ ]:
def load_skeleton_json(json_path):
    """
    Load skeleton JSON.

    Expected structure:
    [
        {
            "keypoints": [
                [x, y, z],
                ...
            ]
        }
    ]
    """
    with open(json_path, "r") as f:
        data = json.load(f)

    if not data:
        raise ValueError(f"Empty skeleton JSON: {json_path}")

    keypoints = np.asarray(data[0]["keypoints"], dtype=np.float32)

    if keypoints.shape != (17, 3):
        raise ValueError(
            f"Unexpected skeleton shape {keypoints.shape} "
            f"in {json_path}"
        )

    return keypoints

In [ ]:
def normalize_skeleton(skeleton):
    """
    Pelvis-center + body-scale normalization.

    Input:
        (17, 3)

    Output:
        (17, 3)
    """

    skeleton = skeleton.astype(np.float32).copy()

    # COCO-17:
    # left hip  = 11
    # right hip = 12
    pelvis = (skeleton[11] + skeleton[12]) / 2.0

    # left shoulder  = 5
    # right shoulder = 6
    shoulder_center = (
        (skeleton[5] + skeleton[6]) / 2.0
    )

    # Move pelvis to origin
    skeleton -= pelvis

    # Body scale
    scale = np.linalg.norm(shoulder_center - pelvis)
    scale = max(scale, 1e-6)

    skeleton /= scale

    return skeleton


def compute_bone_vectors(skeleton):
    """
    Compute bone vectors relative to parent joints.

    Input:
        (17, 3)

    Output:
        (17, 3)
    """

    bones = np.zeros_like(skeleton)

    for joint_idx, parent_idx in enumerate(SKELETON_PARENTS):
        if parent_idx >= 0:
            bones[joint_idx] = (
                skeleton[joint_idx] - skeleton[parent_idx]
            )

    return bones

In [ ]:
def temporal_resample(sequence, target_length=64):
    """
    Resample a temporal sequence to target_length.

    Input:
        (T, ...)
    Output:
        (target_length, ...)
    """

    sequence = np.asarray(sequence, dtype=np.float32)

    T = sequence.shape[0]

    if T == target_length:
        return sequence

    if T == 1:
        return np.repeat(
            sequence,
            target_length,
            axis=0
        )

    old_indices = np.linspace(
        0,
        T - 1,
        T
    )

    new_indices = np.linspace(
        0,
        T - 1,
        target_length
    )

    # Flatten everything except time
    flat = sequence.reshape(T, -1)

    resampled = np.empty(
        (target_length, flat.shape[1]),
        dtype=np.float32
    )

    for i in range(flat.shape[1]):
        resampled[:, i] = np.interp(
            new_indices,
            old_indices,
            flat[:, i]
        )

    return resampled.reshape(
        target_length,
        *sequence.shape[1:]
    )

In [ ]:
def extract_skeleton_features(trial_dir):
    """
    Extract the 12 features per joint used by the
    previous Skeleton experiments.

    Features:
        1. normalized XYZ       -> 3
        2. velocity             -> 3
        3. acceleration         -> 3
        4. bone vectors         -> 3

    Total = 12 features/joint.

    Output:
        (64, 17, 12)
    """

    predictions_dir = trial_dir / "predictions"

    json_files = sorted(
        predictions_dir.glob("*.json")
    )

    if len(json_files) == 0:
        raise FileNotFoundError(
            f"No skeleton JSON files found in {predictions_dir}"
        )

    skeleton_sequence = []

    for json_path in json_files:
        skeleton = load_skeleton_json(json_path)
        skeleton = normalize_skeleton(skeleton)
        skeleton_sequence.append(skeleton)

    skeleton_sequence = np.stack(
        skeleton_sequence,
        axis=0
    )  # (T, 17, 3)

    # Velocity
    velocity = np.diff(
        skeleton_sequence,
        axis=0,
        prepend=skeleton_sequence[0:1]
    )

    # Acceleration
    acceleration = np.diff(
        velocity,
        axis=0,
        prepend=velocity[0:1]
    )

    # Bone vectors
    bone_vectors = np.stack(
        [
            compute_bone_vectors(frame)
            for frame in skeleton_sequence
        ],
        axis=0
    )

    # Concatenate:
    # XYZ + velocity + acceleration + bones
    features = np.concatenate(
        [
            skeleton_sequence,
            velocity,
            acceleration,
            bone_vectors
        ],
        axis=-1
    )

    # (T, 17, 12)
    features = temporal_resample(
        features,
        SKELETON_SEQUENCE_LENGTH
    )

    return features.astype(np.float32)

In [ ]:
class SkeletonDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        trial_dir = (
            SKELETON_ROOT
            / row["action"]
            / row["user"]
            / row["trial"]
        )

        features = extract_skeleton_features(
            trial_dir
        )

        label = int(row["action"].split("_")[0])

        return (
            torch.from_numpy(features),
            torch.tensor(label, dtype=torch.long)
        )

## F. — Skeleton Dataset sanity check


In [ ]:
skeleton_train_dataset = SkeletonDataset(f0_train_df)
skeleton_val_dataset = SkeletonDataset(f0_val_df)

x, y = skeleton_train_dataset[0]

print("Skeleton sample shape:", x.shape)
print("Skeleton label:", y.item())
print("dtype:", x.dtype)
print("min:", x.min().item())
print("max:", x.max().item())
print("mean:", x.mean().item())
print("std:", x.std().item())

## Z. Skeleton DataLoaders


In [ ]:
skeleton_train_loader = DataLoader(
    skeleton_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

skeleton_val_loader = DataLoader(
    skeleton_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Skeleton train batches:", len(skeleton_train_loader))
print("Skeleton val batches:", len(skeleton_val_loader))

x_batch, y_batch = next(iter(skeleton_train_loader))

print("X batch shape:", x_batch.shape)
print("Y batch shape:", y_batch.shape)
print("X dtype:", x_batch.dtype)
print("Y dtype:", y_batch.dtype)
print("X device:", x_batch.device)
print("Labels:", y_batch[:10].tolist())

In [ ]:
# ============================================================
# CELL 13 — S6 Skeleton Encoder
# ============================================================

class S6SkeletonEncoder(nn.Module):

    def __init__(
        self,
        input_size=204,
        projection_size=128,
        hidden_size=128,
        num_layers=2,
        num_heads=4,
        dropout=0.3
    ):
        super().__init__()

        # 17 joints × 12 features = 204
        self.input_projection = nn.Sequential(
            nn.Linear(input_size, projection_size),
            nn.LayerNorm(projection_size),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.lstm = nn.LSTM(
            input_size=projection_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        # BiLSTM output = 128 × 2 = 256
        feature_size = hidden_size * 2

        self.temporal_attention = nn.MultiheadAttention(
            embed_dim=feature_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(feature_size)

        # Learned temporal/frame attention
        self.frame_attention = nn.Sequential(
            nn.Linear(feature_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        # This produces the representation used for fusion
        self.embedding_size = feature_size

    def encode(self, x):
        """
        Input:
            [B, 64, 17, 12]

        Output:
            [B, 256]
        """

        B, T, J, F = x.shape

        # Flatten joints/features
        x = x.reshape(B, T, J * F)

        # [B, 64, 204] -> [B, 64, 128]
        x = self.input_projection(x)

        # -> [B, 64, 256]
        x, _ = self.lstm(x)

        # Temporal self-attention
        attended, _ = self.temporal_attention(
            x, x, x
        )

        # Residual connection + normalization
        x = self.norm(x + attended)

        # Learn importance of each frame
        scores = self.frame_attention(x)

        weights = torch.softmax(
            scores,
            dim=1
        )

        # Weighted temporal pooling
        x = torch.sum(
            x * weights,
            dim=1
        )

        return x

    def forward(self, x):
        return self.encode(x)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

skeleton_encoder = S6SkeletonEncoder().to(device)

x_batch, y_batch = next(iter(skeleton_train_loader))

x_batch = x_batch.to(device)

with torch.no_grad():
    skeleton_embedding = skeleton_encoder(x_batch)

print("Input shape:", x_batch.shape)
print("Embedding shape:", skeleton_embedding.shape)
print("Embedding dtype:", skeleton_embedding.dtype)

# 06. IMU Dataset

In [ ]:
# ============================================================
# CELL 15 — I4 IMU Configuration
# ============================================================

IMU_SEQUENCE_LENGTH = 64

IMU_SENSOR_ORDER = [
    "WTLA",   # Left Arm
    "WTRA",   # Right Arm
    "WTC",    # Chest
    "WTLL",   # Left Leg
    "WTRL"    # Right Leg
]

IMU_FEATURE_COLUMNS = [
    "加速度X(g)",
    "加速度Y(g)",
    "加速度Z(g)",
    "角速度X(°/s)",
    "角速度Y(°/s)",
    "角速度Z(°/s)"
]

IMU_INPUT_SIZE = len(IMU_SENSOR_ORDER) * len(IMU_FEATURE_COLUMNS)

print("Sensors:", IMU_SENSOR_ORDER)
print("Features per sensor:", len(IMU_FEATURE_COLUMNS))
print("Total features per timestep:", IMU_INPUT_SIZE)

## B. Load and synchronize one IMU trial

In [ ]:
def load_imu_trial(trial_dir):
    """
    Load the five IMU sensors from one trial.

    Returns:
        {
            sensor_name: DataFrame
        }
    """

    up_file = trial_dir / "up(LA+RA+C).csv"
    down_file = trial_dir / "down(LL+RL).csv"

    if not up_file.exists() or not down_file.exists():
        raise FileNotFoundError(
            f"Missing IMU files in {trial_dir}"
        )

    up_df = pd.read_csv(up_file)
    down_df = pd.read_csv(down_file)

    df = pd.concat(
        [up_df, down_df],
        ignore_index=True
    )

    # Parse timestamps
    df["时间"] = pd.to_datetime(
        df["时间"],
        errors="coerce"
    )

    # Remove invalid rows
    df = df.dropna(
        subset=["时间", "设备名称"]
    )

    sensor_data = {}

    for sensor in IMU_SENSOR_ORDER:

        sensor_df = df[
            df["设备名称"].astype(str).str.startswith(sensor)
        ].copy()

        sensor_df = sensor_df.sort_values(
            "时间"
        )

        sensor_df = sensor_df[
            ["时间"] + IMU_FEATURE_COLUMNS
        ].copy()

        sensor_df[IMU_FEATURE_COLUMNS] = (
            sensor_df[IMU_FEATURE_COLUMNS]
            .apply(pd.to_numeric, errors="coerce")
        )

        sensor_df = sensor_df.dropna(
            subset=IMU_FEATURE_COLUMNS
        )

        sensor_data[sensor] = sensor_df

    return sensor_data

## C. IMU synchronization

In [ ]:
def synchronize_imu(
    sensor_data,
    sequence_length=64
):
    """
    Synchronize all five IMU sensors onto a common
    temporal grid and resample to 64 frames.

    Output:
        (64, 30)
    """

    # Find the common temporal overlap
    start_time = max(
        df["时间"].min()
        for df in sensor_data.values()
        if len(df) > 0
    )

    end_time = min(
        df["时间"].max()
        for df in sensor_data.values()
        if len(df) > 0
    )

    if end_time <= start_time:
        raise ValueError(
            "No common temporal overlap between IMU sensors."
        )

    # Common timeline
    target_times = pd.date_range(
        start=start_time,
        end=end_time,
        periods=sequence_length
    )

    target_seconds = (
        target_times.astype("int64") / 1e9
    ).to_numpy()

    all_sensor_features = []

    for sensor in IMU_SENSOR_ORDER:

        df = sensor_data[sensor]

        if len(df) < 2:
            raise ValueError(
                f"Sensor {sensor} has insufficient data."
            )

        source_seconds = (
            df["时间"].astype("int64") / 1e9
        ).to_numpy()

        sensor_features = []

        for feature in IMU_FEATURE_COLUMNS:

            values = (
                df[feature]
                .astype(np.float32)
                .to_numpy()
            )

            interpolated = np.interp(
                target_seconds,
                source_seconds,
                values
            )

            sensor_features.append(
                interpolated
            )

        # (64, 6)
        sensor_features = np.stack(
            sensor_features,
            axis=1
        )

        all_sensor_features.append(
            sensor_features
        )

    # (64, 5, 6)
    synchronized = np.stack(
        all_sensor_features,
        axis=1
    )

    # Flatten sensors
    # (64, 5, 6) -> (64, 30)
    synchronized = synchronized.reshape(
        sequence_length,
        -1
    )

    return synchronized.astype(np.float32)

In [ ]:
# ============================================================
# CELL 18 — IMU synchronization sanity check
# ============================================================

sample_row = f0_train_df.iloc[0]

sample_imu_trial = (
    IMU_ROOT
    / sample_row["action"]
    / sample_row["user"]
    / sample_row["trial"]
)

sensor_data = load_imu_trial(
    sample_imu_trial
)

print("Trial:", sample_imu_trial)
print()

for sensor in IMU_SENSOR_ORDER:
    print(
        sensor,
        "rows:",
        len(sensor_data[sensor])
    )

imu_sample = synchronize_imu(
    sensor_data,
    IMU_SEQUENCE_LENGTH
)

print()
print("IMU sample shape:", imu_sample.shape)
print("dtype:", imu_sample.dtype)
print("min:", np.min(imu_sample))
print("max:", np.max(imu_sample))
print("mean:", np.mean(imu_sample))
print("std:", np.std(imu_sample))

## D. IMU training normalization statistics

In [ ]:
print("Computing IMU normalization statistics...")
print("Training trials:", len(f0_train_df))

# We calculate statistics per sensor × feature.
# Shape: (5 sensors, 6 features)

imu_sum = np.zeros(
    (len(IMU_SENSOR_ORDER), len(IMU_FEATURE_COLUMNS)),
    dtype=np.float64
)

imu_sum_sq = np.zeros_like(imu_sum)
imu_count = np.zeros_like(imu_sum, dtype=np.int64)

for idx, row in f0_train_df.iterrows():

    trial_dir = (
        IMU_ROOT
        / row["action"]
        / row["user"]
        / row["trial"]
    )

    try:
        sensor_data = load_imu_trial(trial_dir)
        sequence = synchronize_imu(
            sensor_data,
            IMU_SEQUENCE_LENGTH
        )

        # (64, 30) -> (64, 5, 6)
        sequence = sequence.reshape(
            IMU_SEQUENCE_LENGTH,
            len(IMU_SENSOR_ORDER),
            len(IMU_FEATURE_COLUMNS)
        )

        valid = np.isfinite(sequence)

        imu_sum += np.where(
            valid,
            sequence,
            0
        ).sum(axis=0)

        imu_sum_sq += np.where(
            valid,
            sequence ** 2,
            0
        ).sum(axis=0)

        imu_count += valid.sum(axis=0)

    except Exception as e:
        continue

    if (idx + 1) % 500 == 0:
        print(
            f"Processed {idx + 1}/{len(f0_train_df)}"
        )

imu_mean = imu_sum / np.maximum(imu_count, 1)

imu_variance = (
    imu_sum_sq / np.maximum(imu_count, 1)
    - imu_mean ** 2
)

imu_variance = np.maximum(
    imu_variance,
    1e-8
)

imu_std = np.sqrt(imu_variance)

print("\nNormalization statistics computed.")
print("Mean shape:", imu_mean.shape)
print("Std shape:", imu_std.shape)

print("\nMean:")
print(imu_mean)

print("\nStd:")
print(imu_std)

In [ ]:
# ============================================================
# CELL 23 — Keep only valid IMU trials
# ============================================================

def is_valid_imu_trial(row):
    trial_dir = (
        IMU_ROOT
        / row["action"]
        / row["user"]
        / row["trial"]
    )

    try:
        sensor_data = load_imu_trial(trial_dir)

        # Every sensor must have enough samples
        for sensor in IMU_SENSOR_ORDER:
            if len(sensor_data[sensor]) < 2:
                return False

        # Synchronization must have a common overlap
        _ = synchronize_imu(
            sensor_data,
            IMU_SEQUENCE_LENGTH
        )

        return True

    except Exception:
        return False


print("Checking IMU training trials...")

train_valid_mask = []

for i, (_, row) in enumerate(
    f0_train_df.iterrows()
):

    valid = is_valid_imu_trial(row)
    train_valid_mask.append(valid)

    if (i + 1) % 500 == 0:
        print(
            f"Checked {i + 1}/{len(f0_train_df)}"
        )

valid_imu_train_df = f0_train_df[
    train_valid_mask
].reset_index(drop=True)


print()
print("Original IMU train trials:",
      len(f0_train_df))

print("Valid IMU train trials:",
      len(valid_imu_train_df))

print("Removed IMU train trials:",
      len(f0_train_df) - len(valid_imu_train_df))
# ============================================================
# CELL 24 — Keep only valid IMU validation trials
# ============================================================

print("Checking IMU validation trials...")

val_valid_mask = []

for i, (_, row) in enumerate(
    f0_val_df.iterrows()
):

    valid = is_valid_imu_trial(row)
    val_valid_mask.append(valid)

    if (i + 1) % 200 == 0:
        print(
            f"Checked {i + 1}/{len(f0_val_df)}"
        )

valid_imu_val_df = f0_val_df[
    val_valid_mask
].reset_index(drop=True)


print()
print("Original IMU val trials:",
      len(f0_val_df))

print("Valid IMU val trials:",
      len(valid_imu_val_df))

print("Removed IMU val trials:",
      len(f0_val_df) - len(valid_imu_val_df))

## E. I4 Normalized IMU Dataset

In [ ]:
class I4IMUDataset(Dataset):

    def __init__(
        self,
        dataframe,
        imu_mean,
        imu_std
    ):
        self.df = dataframe.reset_index(drop=True)

        # Store normalization statistics
        # Shape: (5, 6)
        self.mean = imu_mean.astype(np.float32)
        self.std = imu_std.astype(np.float32)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        trial_dir = (
            IMU_ROOT
            / row["action"]
            / row["user"]
            / row["trial"]
        )

        try:

            sensor_data = load_imu_trial(
                trial_dir
            )

            sequence = synchronize_imu(
                sensor_data,
                IMU_SEQUENCE_LENGTH
            )

        except Exception as e:

            raise RuntimeError(
                f"Failed to load IMU trial:\n"
                f"{trial_dir}\n"
                f"Error: {e}"
            )

        # ----------------------------------------------------
        # Reshape:
        # (64, 30) -> (64, 5, 6)
        # ----------------------------------------------------

        sequence = sequence.reshape(
            IMU_SEQUENCE_LENGTH,
            len(IMU_SENSOR_ORDER),
            len(IMU_FEATURE_COLUMNS)
        )

        # ----------------------------------------------------
        # Sensor-wise normalization
        # ----------------------------------------------------

        sequence = (
            sequence - self.mean[None, :, :]
        ) / (
            self.std[None, :, :] + 1e-6
        )

        # ----------------------------------------------------
        # Flatten:
        # (64, 5, 6) -> (64, 30)
        # ----------------------------------------------------

        sequence = sequence.reshape(
            IMU_SEQUENCE_LENGTH,
            IMU_INPUT_SIZE
        )

        # Safety check
        if not np.isfinite(sequence).all():
            raise RuntimeError(
                f"NaN/Inf detected in IMU trial:\n"
                f"{trial_dir}"
            )

        label = int(
            row["action"].split("_")[0]
        )

        return (
            torch.from_numpy(
                sequence.astype(np.float32)
            ),
            torch.tensor(
                label,
                dtype=torch.long
            )
        )

imu_train_dataset = I4IMUDataset(
    valid_imu_train_df,
    imu_mean,
    imu_std
)

imu_val_dataset = I4IMUDataset(
    valid_imu_val_df,
    imu_mean,
    imu_std
)

print(
    "IMU train samples:",
    len(imu_train_dataset)
)

print(
    "IMU val samples:",
    len(imu_val_dataset)
)

## F. CHECK

In [ ]:
imu_x, imu_y = imu_train_dataset[0]

print("IMU sample shape:", imu_x.shape)
print("IMU label:", imu_y.item())
print("dtype:", imu_x.dtype)

print("min:", imu_x.min().item())
print("max:", imu_x.max().item())
print("mean:", imu_x.mean().item())
print("std:", imu_x.std().item())

print(
    "Contains NaN:",
    torch.isnan(imu_x).any().item()
)

print(
    "Contains Inf:",
    torch.isinf(imu_x).any().item()
)

In [ ]:
# ============================================================
# CELL 26 — I4 IMU DataLoaders
# ============================================================

imu_train_loader = DataLoader(
    imu_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

imu_val_loader = DataLoader(
    imu_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print(
    "IMU train batches:",
    len(imu_train_loader)
)

print(
    "IMU val batches:",
    len(imu_val_loader)
)

In [ ]:
# ============================================================
# CELL 28 — I4 IMU Encoder
# ============================================================

class I4IMUEncoder(nn.Module):

    def __init__(
        self,
        input_size=30,
        projection_size=128,
        hidden_size=128,
        num_layers=2,
        num_heads=4,
        dropout=0.3
    ):
        super().__init__()

        self.input_projection = nn.Sequential(
            nn.Linear(input_size, projection_size),
            nn.LayerNorm(projection_size),
            nn.GELU(),
            nn.Dropout(dropout)
        )

        self.lstm = nn.LSTM(
            input_size=projection_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0,
            bidirectional=True
        )

        # BiLSTM:
        # 128 hidden × 2 directions = 256
        feature_size = hidden_size * 2

        self.temporal_attention = nn.MultiheadAttention(
            embed_dim=feature_size,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm = nn.LayerNorm(feature_size)

        self.frame_attention = nn.Sequential(
            nn.Linear(feature_size, 128),
            nn.Tanh(),
            nn.Linear(128, 1)
        )

        self.embedding_size = feature_size

    def encode(self, x):
        """
        Input:
            [B, 64, 30]

        Output:
            [B, 256]
        """

        # [B, 64, 30]
        x = self.input_projection(x)

        # [B, 64, 256]
        x, _ = self.lstm(x)

        # Temporal self-attention
        attended, _ = self.temporal_attention(
            x, x, x
        )

        # Residual + LayerNorm
        x = self.norm(
            x + attended
        )

        # Learned frame importance
        scores = self.frame_attention(x)

        weights = torch.softmax(
            scores,
            dim=1
        )

        # Weighted temporal pooling
        x = torch.sum(
            x * weights,
            dim=1
        )

        return x

    def forward(self, x):
        return self.encode(x)

imu_encoder = I4IMUEncoder().to(device)

imu_x_batch, imu_y_batch = next(
    iter(imu_train_loader)
)

imu_x_batch = imu_x_batch.to(device)

with torch.no_grad():
    imu_embedding = imu_encoder(
        imu_x_batch
    )

print("Input shape:", imu_x_batch.shape)
print("Embedding shape:", imu_embedding.shape)
print("Embedding dtype:", imu_embedding.dtype)

# 07. IR Dataset

In [ ]:
def load_ir_trial(
    trial_dir,
    sequence_length=SEQUENCE_LENGTH,
    image_height=VISUAL_HEIGHT,
    image_width=VISUAL_WIDTH
):
    png_files = list(trial_dir.glob("*.png"))

    if len(png_files) == 0:
        raise ValueError(f"No IR images found: {trial_dir}")

    # IR filenames contain timestamps/frame IDs.
    # Numeric filename sorting is preferable to arbitrary glob order.
    def sort_key(path):
        return path.name

    png_files = sorted(png_files, key=sort_key)

    frames = []

    for image_path in png_files:

        image = Image.open(image_path).convert("L")

        image = image.resize(
            (image_width, image_height),
            Image.BILINEAR
        )

        image = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        # [H,W] -> [1,H,W]
        image = image[None, :, :]

        frames.append(image)

    frames = np.stack(
        frames,
        axis=0
    )

    # frames = [T,C,H,W]

    T = frames.shape[0]

    if T != sequence_length:

        old_indices = np.linspace(
            0,
            T - 1,
            T
        )

        new_indices = np.linspace(
            0,
            T - 1,
            sequence_length
        )

        resampled = np.zeros(
            (
                sequence_length,
                1,
                image_height,
                image_width
            ),
            dtype=np.float32
        )

        for c in range(1):

            for h in range(image_height):

                for w in range(image_width):

                    resampled[:, c, h, w] = np.interp(
                        new_indices,
                        old_indices,
                        frames[:, c, h, w]
                    )

        frames = resampled

    return frames.astype(np.float32)

In [ ]:
def load_ir_trial(
    trial_dir,
    sequence_length=SEQUENCE_LENGTH,
    image_height=VISUAL_HEIGHT,
    image_width=VISUAL_WIDTH
):
    png_files = sorted(
        trial_dir.glob("*.png"),
        key=lambda p: p.name
    )

    if len(png_files) == 0:
        raise ValueError(
            f"No IR images found: {trial_dir}"
        )

    frames = []

    for image_path in png_files:

        image = Image.open(image_path).convert("L")

        image = image.resize(
            (image_width, image_height),
            Image.BILINEAR
        )

        image = np.asarray(
            image,
            dtype=np.float32
        ) / 255.0

        frames.append(image)

    frames = np.stack(
        frames,
        axis=0
    )

    # [T,H,W]

    if frames.shape[0] != sequence_length:

        tensor = torch.from_numpy(
            frames
        ).unsqueeze(0).unsqueeze(1)

        tensor = F.interpolate(
            tensor,
            size=(
                sequence_length,
                image_height,
                image_width
            ),
            mode="trilinear",
            align_corners=True
        )

        frames = tensor[
            0, 0
        ].numpy()

    # [T,H,W] -> [T,1,H,W]

    frames = frames[:, None, :, :]

    return frames.astype(np.float32)

In [ ]:
sample_row = f0_train_paired_df.iloc[0]

ir_dir = (
    IR_ROOT
    / sample_row["action"]
    / sample_row["user"]
    / sample_row["trial"]
)

ir_sample = load_ir_trial(ir_dir)

print("IR shape:", ir_sample.shape)
print("dtype:", ir_sample.dtype)
print("min:", ir_sample.min())
print("max:", ir_sample.max())
print("mean:", ir_sample.mean())
print("std:", ir_sample.std())

# 08. Depth_Color Dataset

# 09. Skeleton Encoder

## 0. Individual modality classifier


In [ ]:
class ModalityClassifier(nn.Module):

    def __init__(
        self,
        encoder,
        embedding_size=256,
        num_classes=40
    ):
        super().__init__()

        self.encoder = encoder

        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(embedding_size, 128),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):

        embedding = self.encoder.encode(x)

        logits = self.classifier(
            embedding
        )

        return logits


skeleton_model = ModalityClassifier(
    encoder=S6SkeletonEncoder(),
    embedding_size=256,
    num_classes=NUM_CLASSES
).to(device)

imu_model = ModalityClassifier(
    encoder=I4IMUEncoder(),
    embedding_size=256,
    num_classes=NUM_CLASSES
).to(device)

print("Skeleton parameters:",sum(p.numel() for p in skeleton_model.parameters()))

print("IMU parameters:",sum(p.numel() for p in imu_model.parameters()))

# ============================================================
# CELL 33 — Checkpoint directory
# ============================================================

CHECKPOINT_DIR = Path(
    "/kaggle/working/cuhk_x_checkpoints"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(
    "Checkpoint directory:",
    CHECKPOINT_DIR
)

In [ ]:
def train_modality_model(
    model,
    train_loader,
    val_loader,
    run_name,
    checkpoint_path,
    epochs=30,
    learning_rate=1e-3,
    weight_decay=1e-4,
):
    """
    Train one modality model.

    Features:
      - CrossEntropyLoss
      - AdamW optimizer
      - W&B logging
      - best validation checkpoint
      - validation accuracy
      - learning-rate logging
    """

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    # Reduce LR when validation accuracy stops improving
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=3
    )

    wandb.init(
        project="CIUX",
        name=run_name,
        config={
            "model": run_name,
            "epochs": epochs,
            "batch_size": BATCH_SIZE,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "num_classes": NUM_CLASSES,
            "device": str(device)
        },
        reinit="finish_previous"
    )

    best_val_accuracy = -1.0
    best_epoch = -1

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": [],
        "learning_rate": []
    }

    for epoch in range(1, epochs + 1):
        # TRAIN
        
        model.train()

        train_loss_sum = 0.0
        train_correct = 0
        train_total = 0

        train_start = time.time()

        for x, y in train_loader:

            x = x.to(
                device,
                non_blocking=True
            )

            y = y.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(x)

            loss = criterion(
                logits,
                y
            )

            loss.backward()

            # Prevent occasional exploding gradients
            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            train_loss_sum += (
                loss.item() * y.size(0)
            )

            predictions = logits.argmax(
                dim=1
            )

            train_correct += (
                predictions == y
            ).sum().item()

            train_total += y.size(0)

        train_loss = (
            train_loss_sum / train_total
        )

        train_accuracy = (
            train_correct / train_total
        )
        

        # VALIDATION
        model.eval()

        val_loss_sum = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for x, y in val_loader:

                x = x.to(
                    device,
                    non_blocking=True
                )

                y = y.to(
                    device,
                    non_blocking=True
                )

                logits = model(x)

                loss = criterion(
                    logits,
                    y
                )

                val_loss_sum += (
                    loss.item() * y.size(0)
                )

                predictions = logits.argmax(
                    dim=1
                )

                val_correct += (
                    predictions == y
                ).sum().item()

                val_total += y.size(0)

        val_loss = (
            val_loss_sum / val_total
        )

        val_accuracy = (
            val_correct / val_total
        )


        
        # SCHEDULER
        # ====================================================

        scheduler.step(
            val_accuracy
        )

        current_lr = optimizer.param_groups[0]["lr"]

        epoch_time = (
            time.time() - train_start
        )

        # ====================================================
        # SAVE HISTORY
        # ====================================================

        history["train_loss"].append(
            train_loss
        )

        history["train_accuracy"].append(
            train_accuracy
        )

        history["val_loss"].append(
            val_loss
        )

        history["val_accuracy"].append(
            val_accuracy
        )

        history["learning_rate"].append(
            current_lr
        )


        # BEST CHECKPOINT
        # ====================================================

        is_best = (
            val_accuracy > best_val_accuracy
        )

        if is_best:

            best_val_accuracy = val_accuracy
            best_epoch = epoch

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_accuracy": best_val_accuracy,
                    "run_name": run_name
                },
                checkpoint_path
            )


        
        # W&B
        # ====================================================

        wandb.log(
            {
                "epoch": epoch,

                "train/loss": train_loss,
                "train/accuracy": train_accuracy,

                "val/loss": val_loss,
                "val/accuracy": val_accuracy,

                "learning_rate": current_lr,

                "best/val_accuracy": best_val_accuracy,
                "best/epoch": best_epoch,

                "epoch_time_sec": epoch_time
            }
        )


        
        # PRINT
        # ====================================================

        marker = " ★ BEST" if is_best else ""

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy*100:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy*100:.2f}% | "
            f"LR: {current_lr:.2e} | "
            f"Time: {epoch_time:.1f}s"
            f"{marker}"
        )


    # FINISH W&B
    # ========================================================

    wandb.summary["best_val_accuracy"] = (
        best_val_accuracy
    )

    wandb.summary["best_epoch"] = (
        best_epoch
    )

    wandb.finish()

    print()
    print("=" * 70)
    print(f"Training complete: {run_name}")
    print(
        f"Best validation accuracy: "
        f"{best_val_accuracy*100:.2f}%"
    )
    print(
        f"Best epoch: {best_epoch}"
    )
    print(
        f"Checkpoint: {checkpoint_path}"
    )
    print("=" * 70)

    return history

In [ ]:
skeleton_history = train_modality_model(
    model=skeleton_model,
    train_loader=skeleton_train_loader,
    val_loader=skeleton_val_loader,

    run_name="skeleton_s6",

    checkpoint_path=(
        CHECKPOINT_DIR
        / "skeleton_s6_best.pt"
    ),

    epochs=10,
    learning_rate=1e-3,
    weight_decay=1e-4
)

In [ ]:
imu_history = train_modality_model(
    model=imu_model,
    train_loader=imu_train_loader,
    val_loader=imu_val_loader,
    run_name="imu_i4",
    checkpoint_path=CHECKPOINT_DIR / "imu_i4_best.pt",
    epochs=10,
    learning_rate=1e-3,
    weight_decay=1e-4
)

# 13. Fusion Architecture

In [ ]:
# Load best Skeleton checkpoint
skeleton_ckpt = torch.load(
    CHECKPOINT_DIR / "skeleton_s6_best.pt",
    map_location=device
)

skeleton_model.load_state_dict(
    skeleton_ckpt["model_state_dict"]
)

print(
    "Skeleton best epoch:",
    skeleton_ckpt["epoch"],
    "Val Acc:",
    f'{skeleton_ckpt["best_val_accuracy"]:.2%}'
)


# Load best IMU checkpoint
imu_ckpt = torch.load(
    CHECKPOINT_DIR / "imu_i4_best.pt",
    map_location=device
)

imu_model.load_state_dict(
    imu_ckpt["model_state_dict"]
)

print(
    "IMU best epoch:",
    imu_ckpt["epoch"],
    "Val Acc:",
    f'{imu_ckpt["best_val_accuracy"]:.2%}'
)

In [ ]:
def load_skeleton_trial(trial_dir, sequence_length=SEQUENCE_LENGTH):
    """
    Load one Skeleton trial.

    Skeleton JSON files are stored inside:
        <trial>/predictions/

    Returns:
        np.ndarray of shape [64, 17, 12]
    """

    predictions_dir = trial_dir / "predictions"

    json_files = list(
        predictions_dir.glob("*.json")
    )

    if len(json_files) == 0:
        raise ValueError(
            f"No JSON skeleton files found: {predictions_dir}"
        )

    # Sort by filename. The filenames contain frame indices.
    json_files = sorted(
        json_files,
        key=lambda p: p.name
    )

    frames = []

    for json_file in json_files:

        with open(json_file, "r") as f:
            data = json.load(f)

        # -------------------------------------------------
        # Each JSON contains a list with keypoints
        # -------------------------------------------------
        if isinstance(data, list):
            if len(data) == 0:
                continue

            keypoints = data[0]["keypoints"]

        else:
            keypoints = data["keypoints"]

        keypoints = np.asarray(
            keypoints,
            dtype=np.float32
        )

        # [17, 3]
        keypoints = keypoints.reshape(17, 3)

        frames.append(keypoints)

    if len(frames) == 0:
        raise ValueError(
            f"No valid skeleton frames found: {predictions_dir}"
        )

    skeleton = np.stack(
        frames,
        axis=0
    )

    # =====================================================
    # Pelvis-centered normalization
    # =====================================================

    pelvis = (
        skeleton[:, 11, :] +
        skeleton[:, 12, :]
    ) / 2.0

    skeleton = (
        skeleton -
        pelvis[:, None, :]
    )

    # =====================================================
    # Scale normalization
    # =====================================================

    shoulder_center = (
        skeleton[:, 5, :] +
        skeleton[:, 6, :]
    ) / 2.0

    scale = np.linalg.norm(
        shoulder_center,
        axis=1,
        keepdims=True
    )

    scale = np.clip(
        scale,
        1e-6,
        None
    )

    skeleton = (
        skeleton /
        scale[:, None, :]
    )

    # =====================================================
    # Velocity
    # =====================================================

    velocity = np.diff(
        skeleton,
        axis=0,
        prepend=skeleton[0:1]
    )

    # =====================================================
    # Acceleration
    # =====================================================

    acceleration = np.diff(
        velocity,
        axis=0,
        prepend=velocity[0:1]
    )

    # =====================================================
    # Bone vectors
    # =====================================================

    parents = [
        -1, 0, 0, 1, 2,
        11, 12,
        5, 6,
        7, 8,
        -1, -1,
        11, 12,
        13, 14
    ]

    bone_vectors = np.zeros_like(
        skeleton
    )

    for joint, parent in enumerate(parents):

        if parent >= 0:

            bone_vectors[:, joint, :] = (
                skeleton[:, joint, :]
                - skeleton[:, parent, :]
            )

    # =====================================================
    # Combine
    # =====================================================

    features = np.concatenate(
        [
            skeleton,       # 3
            velocity,       # 3
            acceleration,   # 3
            bone_vectors    # 3
        ],
        axis=-1
    )

    # [T, 17, 12]
    assert features.shape[1:] == (
        17, 12
    )

    # =====================================================
    # Temporal resampling
    # =====================================================

    T = features.shape[0]

    if T != sequence_length:

        old_indices = np.linspace(
            0,
            T - 1,
            T
        )

        new_indices = np.linspace(
            0,
            T - 1,
            sequence_length
        )

        resampled = np.zeros(
            (
                sequence_length,
                17,
                12
            ),
            dtype=np.float32
        )

        for joint in range(17):

            for feature in range(12):

                resampled[:, joint, feature] = np.interp(
                    new_indices,
                    old_indices,
                    features[:, joint, feature]
                )

        features = resampled

    return features.astype(
        np.float32
    )
    
f0_train_paired_df = valid_imu_train_df.copy()
f0_val_paired_df = valid_imu_val_df.copy()

print("F0 paired train:", len(f0_train_paired_df))
print("F0 paired val:  ", len(f0_val_paired_df))

print(
    "Train users:",
    sorted(f0_train_paired_df["user"].unique())
)

print(
    "Val users:",
    sorted(f0_val_paired_df["user"].unique())
)

In [ ]:
class F0SkeletonIMUFusion(nn.Module):

    def __init__(
        self,
        skeleton_encoder,
        imu_encoder,
        num_classes=40
    ):
        super().__init__()

        self.skeleton_encoder = skeleton_encoder
        self.imu_encoder = imu_encoder

        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(0.3),

            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.2),

            nn.Linear(128, num_classes)
        )

    def forward(self, skeleton, imu):

        s = self.skeleton_encoder.encode(
            skeleton
        )

        i = self.imu_encoder.encode(
            imu
        )

        fused = torch.cat(
            [s, i],
            dim=1
        )

        return self.classifier(fused)

class SkeletonIMUDataset(Dataset):

    def __init__(self, df):
        self.df = df.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        action = row["action"]
        user = row["user"]
        trial = row["trial"]

        # =====================================================
        # Skeleton
        # =====================================================
        skeleton_dir = (
            SKELETON_ROOT
            / action
            / user
            / trial
        )

        skeleton = load_skeleton_trial(skeleton_dir)

        skeleton = torch.tensor(
            skeleton,
            dtype=torch.float32
        )

        # Expected:
        # [64, 17, 12]
        assert skeleton.shape == (64, 17, 12), (
            f"Unexpected skeleton shape: {skeleton.shape}"
        )

        # =====================================================
        # IMU
        # =====================================================
        imu_dir = (
            IMU_ROOT
            / action
            / user
            / trial
        )

        sensor_data = load_imu_trial(imu_dir)

        imu = synchronize_imu(
            sensor_data,
            sequence_length=IMU_SEQUENCE_LENGTH
        )

        # [64, 30]
        imu = imu.reshape(
            IMU_SEQUENCE_LENGTH,
            5,
            6
        )

        # Training-set normalization
        imu = (
            imu - IMU_MEAN
        ) / IMU_STD

        imu = imu.reshape(
            IMU_SEQUENCE_LENGTH,
            IMU_INPUT_SIZE
        )

        imu = torch.tensor(
            imu,
            dtype=torch.float32
        )

        assert imu.shape == (64, 30), (
            f"Unexpected IMU shape: {imu.shape}"
        )

        # =====================================================
        # Label
        # =====================================================
        label = int(action.split("_")[0])

        label = torch.tensor(
            label,
            dtype=torch.long
        )

        return skeleton, imu, label

In [ ]:
f0_train_dataset = SkeletonIMUDataset(
    f0_train_paired_df
)

f0_val_dataset = SkeletonIMUDataset(
    f0_val_paired_df
)

print("F0 train:", len(f0_train_dataset))
print("F0 val:  ", len(f0_val_dataset))

In [ ]:
IMU_MEAN = np.array([
    [-0.34742488, 0.20479452, 0.26163209, -0.29040349, -1.67002889, -0.98411671],
    [ 0.33568027, 0.20783301, 0.32133636,  0.90558618,  2.67798786,  2.71813569],
    [-0.04481415, 0.79413201, -0.10782608, -1.28675364,  1.48912781, -0.13205434],
    [-0.05770191, 0.78932931, -0.05539985, -0.28900713,  0.70100786, -0.11024557],
    [ 0.05343975, 0.84168749, -0.08283601, -0.60211366, -0.00606721, -0.66025915]
], dtype=np.float32)

IMU_STD = np.array([
    [0.60299489, 0.71772565, 0.53411586, 69.30133874, 103.23974317, 86.82104813],
    [0.61559026, 0.71588514, 0.52497034, 71.68974573, 104.10524373, 86.58482081],
    [0.16835922, 0.49538102, 0.34679285, 28.14689793, 37.38931142, 15.24167692],
    [0.28100786, 0.57683786, 0.32819254, 50.51455883, 50.18963781, 27.41363781],
    [0.32276691, 0.46834463, 0.38813124, 50.36168425, 57.77715868, 38.84884282]
], dtype=np.float32)

print("IMU_MEAN:", IMU_MEAN.shape)
print("IMU_STD: ", IMU_STD.shape)
print("Minimum STD:", IMU_STD.min())
skeleton_x, imu_x, y = f0_train_dataset[0]

print("Skeleton:", skeleton_x.shape, skeleton_x.dtype)
print("IMU:     ", imu_x.shape, imu_x.dtype)
print("Label:   ", y.item())

print("\nSkeleton:")
print("  min:", skeleton_x.min().item())
print("  max:", skeleton_x.max().item())
print("  mean:", skeleton_x.mean().item())
print("  std:", skeleton_x.std().item())

print("\nIMU:")
print("  min:", imu_x.min().item())
print("  max:", imu_x.max().item())
print("  mean:", imu_x.mean().item())
print("  std:", imu_x.std().item())

print("\nNaN:")
print("  Skeleton:", torch.isnan(skeleton_x).any().item())
print("  IMU:     ", torch.isnan(imu_x).any().item())

print("\nInf:")
print("  Skeleton:", torch.isinf(skeleton_x).any().item())
print("  IMU:     ", torch.isinf(imu_x).any().item())

In [ ]:
f0_train_loader = DataLoader(
    f0_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

f0_val_loader = DataLoader(
    f0_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

print("F0 train batches:", len(f0_train_loader))
print("F0 val batches:  ", len(f0_val_loader))

In [ ]:
skeleton_batch, imu_batch, labels = next(
    iter(f0_train_loader)
)

print("Skeleton batch:", skeleton_batch.shape)
print("IMU batch:     ", imu_batch.shape)
print("Labels:        ", labels.shape)

print("Skeleton dtype:", skeleton_batch.dtype)
print("IMU dtype:     ", imu_batch.dtype)
print("Labels dtype:  ", labels.dtype)

In [ ]:
class F0V2GatedFusion(nn.Module):

    def __init__(
        self,
        skeleton_encoder,
        imu_encoder,
        embedding_size=256,
        num_classes=40,
        dropout=0.3
    ):
        super().__init__()

        self.skeleton_encoder = skeleton_encoder
        self.imu_encoder = imu_encoder

        # Learn a gate for each modality.
        # Values are between 0 and 1.
        self.gate = nn.Sequential(
            nn.Linear(
                embedding_size * 2,
                128
            ),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, 2)
        )

        # Fusion classifier
        self.classifier = nn.Sequential(
            nn.Linear(
                embedding_size,
                256
            ),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.GELU(),
            nn.Dropout(0.2),

            nn.Linear(128, num_classes)
        )

    def forward(
        self,
        skeleton,
        imu,
        return_gates=False
    ):

        # ---------------------------------------------
        # Extract modality embeddings
        # ---------------------------------------------

        skeleton_embedding = (
            self.skeleton_encoder.encode(
                skeleton
            )
        )

        imu_embedding = (
            self.imu_encoder.encode(
                imu
            )
        )

        # ---------------------------------------------
        # Concatenate only for gate prediction
        # ---------------------------------------------

        combined = torch.cat(
            [
                skeleton_embedding,
                imu_embedding
            ],
            dim=1
        )

        gate_logits = self.gate(
            combined
        )

        gates = torch.softmax(
            gate_logits,
            dim=1
        )

        skeleton_gate = gates[:, 0:1]
        imu_gate = gates[:, 1:2]

        # ---------------------------------------------
        # Weighted fusion
        # ---------------------------------------------

        fused = (
            skeleton_gate * skeleton_embedding
            +
            imu_gate * imu_embedding
        )

        # ---------------------------------------------
        # Classification
        # ---------------------------------------------

        logits = self.classifier(
            fused
        )

        if return_gates:
            return logits, gates

        return logits

In [ ]:
# ---------------------------------------------------------
# Create fresh encoders
# ---------------------------------------------------------

skeleton_encoder = S6SkeletonEncoder().to(device)
imu_encoder = I4IMUEncoder().to(device)


# ---------------------------------------------------------
# Load Skeleton checkpoint
# ---------------------------------------------------------

skeleton_checkpoint = torch.load(
    CHECKPOINT_DIR / "skeleton_s6_best.pt",
    map_location=device
)

skeleton_model_temp = ModalityClassifier(
    skeleton_encoder,
    embedding_size=256,
    num_classes=40
).to(device)

skeleton_model_temp.load_state_dict(
    skeleton_checkpoint["model_state_dict"]
)


# ---------------------------------------------------------
# Load IMU checkpoint
# ---------------------------------------------------------

imu_checkpoint = torch.load(
    CHECKPOINT_DIR / "imu_i4_best.pt",
    map_location=device
)

imu_model_temp = ModalityClassifier(
    imu_encoder,
    embedding_size=256,
    num_classes=40
).to(device)

imu_model_temp.load_state_dict(
    imu_checkpoint["model_state_dict"]
)


print("Skeleton checkpoint loaded.")
print("IMU checkpoint loaded.")

for param in skeleton_encoder.parameters():
    param.requires_grad = False

for param in imu_encoder.parameters():
    param.requires_grad = False

print(
    "Skeleton trainable:",
    sum(p.numel() for p in skeleton_encoder.parameters() if p.requires_grad)
)

print(
    "IMU trainable:",
    sum(p.numel() for p in imu_encoder.parameters() if p.requires_grad)
)

In [ ]:
f0_v2_model = F0V2GatedFusion(
    skeleton_encoder=skeleton_encoder,
    imu_encoder=imu_encoder,
    embedding_size=256,
    num_classes=NUM_CLASSES,
    dropout=0.3
).to(device)

trainable_params = sum(
    p.numel()
    for p in f0_v2_model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in f0_v2_model.parameters()
)

print(
    "Total parameters:",
    f"{total_params:,}"
)

print(
    "Trainable parameters:",
    f"{trainable_params:,}"
)

f0_v2_model.eval()

with torch.no_grad():

    skeleton_batch = skeleton_batch.to(device)
    imu_batch = imu_batch.to(device)

    logits = f0_v2_model(
        skeleton_batch,
        imu_batch
    )

print("Skeleton input:", skeleton_batch.shape)
print("IMU input:     ", imu_batch.shape)
print("Output logits: ", logits.shape)

In [ ]:
def train_f0_fusion(
    model,
    train_loader,
    val_loader,
    run_name,
    checkpoint_path,
    epochs=10,
    learning_rate=1e-3,
    weight_decay=1e-4,
):

    model = model.to(device)

    criterion = nn.CrossEntropyLoss()

    # Only train parameters that require gradients
    optimizer = torch.optim.AdamW(
        filter(
            lambda p: p.requires_grad,
            model.parameters()
        ),
        lr=learning_rate,
        weight_decay=weight_decay
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=0.5,
        patience=2
    )

    wandb.init(
        project="CIUX",
        name=run_name,
        config={
            "architecture": "F0_Skeleton_IMU",
            "epochs": epochs,
            "batch_size": BATCH_SIZE,
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "frozen_encoders": True,
            "skeleton_embedding": 256,
            "imu_embedding": 256,
            "fusion_embedding": 512,
            "num_classes": NUM_CLASSES
        },
        reinit="finish_previous"
    )

    best_val_accuracy = -1.0
    best_epoch = -1

    history = {
        "train_loss": [],
        "train_accuracy": [],
        "val_loss": [],
        "val_accuracy": []
    }

    for epoch in range(1, epochs + 1):

        start_time = time.time()

        # =====================================================
        # TRAIN
        # =====================================================

        model.train()

        # Keep frozen encoders in eval mode so their Dropout
        # does not introduce noise during fusion-head training.
        model.skeleton_encoder.eval()
        model.imu_encoder.eval()

        train_loss = 0.0
        train_correct = 0
        train_total = 0

        for skeleton, imu, labels in train_loader:

            skeleton = skeleton.to(
                device,
                non_blocking=True
            )

            imu = imu.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad()

            logits = model(
                skeleton,
                imu
            )

            loss = criterion(
                logits,
                labels
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                max_norm=1.0
            )

            optimizer.step()

            train_loss += (
                loss.item() *
                labels.size(0)
            )

            predictions = logits.argmax(dim=1)

            train_correct += (
                predictions == labels
            ).sum().item()

            train_total += labels.size(0)

        train_loss /= train_total

        train_accuracy = (
            train_correct /
            train_total
        )

        # =====================================================
        # VALIDATION
        # =====================================================

        model.eval()

        val_loss = 0.0
        val_correct = 0
        val_total = 0

        with torch.no_grad():

            for skeleton, imu, labels in val_loader:

                skeleton = skeleton.to(
                    device,
                    non_blocking=True
                )

                imu = imu.to(
                    device,
                    non_blocking=True
                )

                labels = labels.to(
                    device,
                    non_blocking=True
                )

                logits = model(
                    skeleton,
                    imu
                )

                loss = criterion(
                    logits,
                    labels
                )

                val_loss += (
                    loss.item() *
                    labels.size(0)
                )

                predictions = logits.argmax(dim=1)

                val_correct += (
                    predictions == labels
                ).sum().item()

                val_total += labels.size(0)

        val_loss /= val_total

        val_accuracy = (
            val_correct /
            val_total
        )

        scheduler.step(
            val_accuracy
        )

        current_lr = optimizer.param_groups[0]["lr"]

        # =====================================================
        # BEST CHECKPOINT
        # =====================================================

        is_best = (
            val_accuracy >
            best_val_accuracy
        )

        if is_best:

            best_val_accuracy = val_accuracy
            best_epoch = epoch

            torch.save(
                {
                    "epoch": epoch,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "best_val_accuracy": best_val_accuracy,
                    "run_name": run_name
                },
                checkpoint_path
            )

        # =====================================================
        # HISTORY
        # =====================================================

        history["train_loss"].append(
            train_loss
        )

        history["train_accuracy"].append(
            train_accuracy
        )

        history["val_loss"].append(
            val_loss
        )

        history["val_accuracy"].append(
            val_accuracy
        )

        elapsed = time.time() - start_time

        # =====================================================
        # W&B
        # =====================================================

        wandb.log(
            {
                "epoch": epoch,
                "train/loss": train_loss,
                "train/accuracy": train_accuracy,
                "val/loss": val_loss,
                "val/accuracy": val_accuracy,
                "learning_rate": current_lr,
                "best/val_accuracy": best_val_accuracy,
                "best/epoch": best_epoch,
                "epoch_time_sec": elapsed
            }
        )

        marker = " ★ BEST" if is_best else ""

        print(
            f"Epoch {epoch:02d}/{epochs} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_accuracy:.2%} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_accuracy:.2%} | "
            f"LR: {current_lr:.2e} | "
            f"Time: {elapsed:.1f}s"
            f"{marker}"
        )

    wandb.summary["best_val_accuracy"] = (
        best_val_accuracy
    )

    wandb.summary["best_epoch"] = (
        best_epoch
    )

    wandb.finish()

    print("\n" + "=" * 60)
    print("F0 TRAINING COMPLETE")
    print("=" * 60)
    print(
        f"Best Val Accuracy: "
        f"{best_val_accuracy:.2%}"
    )
    print(
        f"Best Epoch: {best_epoch}"
    )
    print(
        f"Checkpoint: {checkpoint_path}"
    )

    return history

In [ ]:
f0_v2_history = train_f0_fusion(
    model=f0_v2_model,
    train_loader=f0_train_loader,
    val_loader=f0_val_loader,
    run_name="F0_v2_gated_skeleton_imu",
    checkpoint_path=CHECKPOINT_DIR / "F0_v2_gated_best.pt",
    epochs=10,
    learning_rate=1e-3,
    weight_decay=1e-4
)